In [8]:
from transforms3d.quaternions import quat2mat, mat2quat
from transforms3d.affines import compose
import numpy as np
# Align CAD model 
mat_origin = [-0.921508, -0.365140, 0.132274,
              -0.160531, 0.048001, -0.985863,
              0.353629, -0.929714, -0.102849]
transl = [-0.149010,-0.005023,2.092960]
mat = np.array(mat_origin).reshape(-1)
quat = mat2quat(mat)
print(quat)
align_affine = compose(T=transl, R=mat.reshape(3, 3), Z=np.ones(3))
print(align_affine) # Rigid Transform to align robot to points cloud, in robot coordinate

[ 0.07688247  0.18257943 -0.71978427  0.66533032]
[[-0.921508 -0.36514   0.132274 -0.14901 ]
 [-0.160531  0.048001 -0.985863 -0.005023]
 [ 0.353629 -0.929714 -0.102849  2.09296 ]
 [ 0.        0.        0.        1.      ]]


In [9]:
points_cloud_id = [62359, 59662, 58515, 61244]
cad_mesh_id = [1725, 1294, 1438, 1711]

import open3d as o3d

ur_mesh_path = "/remote-home/liuym/data/0721/ur_mesh.obj"
cad_mesh = o3d.io.read_triangle_mesh(ur_mesh_path)

pc_path = "/remote-home/liuym/data/0721/cropped_pc.ply"
pc = o3d.io.read_triangle_mesh(pc_path)


[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).


In [10]:
import numpy as np
cad_pts = np.asarray(cad_mesh.vertices)[cad_mesh_id]
cad_pts_homo = np.concatenate([cad_pts.T, np.ones([1, 4])], axis=0)
print(cad_pts.T)
print(cad_pts_homo)

pc_pts = np.asanyarray(pc.vertices)[points_cloud_id]
pc_homo = np.concatenate([pc_pts.T, np.ones([1, 4])], axis=0)
print(pc_pts.T)
print(pc_homo)

[[-0.001755 -0.062461  0.05437   0.05725 ]
 [-0.094983  0.063986  0.056044 -0.075811]
 [ 0.        0.016299  0.024997  0.      ]]
[[-0.001755 -0.062461  0.05437   0.05725 ]
 [-0.094983  0.063986  0.056044 -0.075811]
 [ 0.        0.016299  0.024997  0.      ]
 [ 1.        1.        1.        1.      ]]
[[-0.13  -0.069 -0.183 -0.152]
 [ 0.005 -0.019 -0.031 -0.003]
 [ 2.017  2.058  2.052  2.025]]
[[-0.13  -0.069 -0.183 -0.152]
 [ 0.005 -0.019 -0.031 -0.003]
 [ 2.017  2.058  2.052  2.025]
 [ 1.     1.     1.     1.   ]]


In [11]:
from transforms3d.quaternions import quat2mat

# cam_base -> cam_body
cam0_body_pose = [-0.003945, -0.031956, -0.000522, 0.495797, -0.503967, 0.504645 , -0.495515]
cam0_transl = np.array(cam0_body_pose[:3])
cam0_quat = np.array(cam0_body_pose[3:]) # w, x, y, z
# cam0_quat = cam0_quat.transpose([-1, 0, 1, 2]) # x, y, z, w
cam0_affine = compose(T=cam0_transl, R=quat2mat(cam0_quat), Z=np.ones(3))

T_cam0_to_rob = align_affine @ pc_homo @ np.linalg.inv(cad_pts_homo)
print(T_cam0_to_rob)

affine = np.linalg.inv(T_cam0_to_rob) @ np.linalg.inv(cam0_affine)
print(affine)

quat = mat2quat(affine[:3, :3]) 
print(quat)
transl = affine[:3, -1]
print(transl)

[[ 5.71413605e-01 -4.93624719e-01  4.36430875e+00  1.89873874e-01]
 [ 6.46718533e-02 -4.46233761e-01  1.44171460e+00 -2.01467076e+00]
 [-1.27302172e-01  3.31032101e-01 -1.26904421e+00  1.86611223e+00]
 [ 1.77635684e-15  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
[[-2.28193822e+01  1.65760920e+00  1.56861453e+01  1.25725628e+01]
 [ 1.00605179e+01 -1.89638882e+00 -3.34415780e+00 -1.28810888e+01]
 [ 4.12551507e+00 -6.60650856e-01 -2.43222763e+00 -3.15385323e+00]
 [ 4.05353657e-14 -2.94450543e-15 -2.78641915e-14  1.00000000e+00]]
[0.21100327 0.40926145 0.43824412 0.77196164]
[ 12.57256284 -12.88108878  -3.15385323]
